In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

CLEAN_DIR = Path("../data/cic2018_clean/")
MODELS_DIR = Path("../models/")

df = pd.read_parquet(CLEAN_DIR / "cic2018_final_sampled.parquet")

# Reuse the saved LabelEncoder so class-to-number mapping matches
# the XGBoost run exactly — critical for a valid comparison
le = joblib.load(MODELS_DIR / "label_encoder.pkl")
df['label_encoded'] = le.transform(df['Label'])

X = df.drop(columns=['Label', 'label_encoded'])
y = df['label_encoded']

# random_state=42 + stratify=y reproduces the IDENTICAL split used
# during XGBoost training, so both models see precisely the same
# held-out test rows — otherwise the comparison would be meaningless
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Test set: {X_test.shape}")
print(f"Benign class index: {list(le.classes_).index('Benign')}")

Test set: (231652, 78)
Benign class index: 0


In [2]:
# Check the actual current state of the file
print(f"Total rows currently in file: {len(df):,}")
print(f"Expected from original sampling: 1,041,092")
print(f"Difference: {len(df) - 1_041_092:,}")

print("\nCurrent label distribution:")
print(df['Label'].value_counts())

print(f"\nDuplicate rows present: {df.duplicated().sum():,}")

Total rows currently in file: 1,158,260
Expected from original sampling: 1,041,092
Difference: 117,168

Current label distribution:
Label
Benign                      150000
Bot                         150000
DoS attacks-Hulk            150000
DDoS attacks-LOIC-HTTP      150000
Infilteration               150000
SSH-Bruteforce              150000
DDOS attack-HOIC            150000
DoS attacks-GoldenEye        82812
DoS attacks-Slowloris        19816
DDOS attack-LOIC-UDP          3460
Brute Force -Web              1140
Brute Force -XSS               458
DoS attacks-SlowHTTPTest       214
FTP-BruteForce                 190
SQL Injection                  170
Name: count, dtype: int64

Duplicate rows present: 304,382


In [3]:
import pandas as pd
import numpy as np
import gc
import joblib
from pathlib import Path

CLEAN_DIR = Path("../data/cic2018_clean/")
MODELS_DIR = Path("../models/")

In [4]:
CAP = 150_000

sampled_dfs = []
for f in sorted(CLEAN_DIR.glob("*_cleaned.parquet")):
    d = pd.read_parquet(f)
    for label in d['Label'].unique():
        sampled_dfs.append(d[d['Label'] == label])
    del d
    gc.collect()

df_combined = pd.concat(sampled_dfs, ignore_index=True)
del sampled_dfs
gc.collect()

print(f"Before capping: {df_combined.shape}")

final_dfs = []
for label in df_combined['Label'].unique():
    subset = df_combined[df_combined['Label'] == label]
    if len(subset) > CAP:
        subset = subset.sample(n=CAP, random_state=42)
    final_dfs.append(subset)

df_regen = pd.concat(final_dfs, ignore_index=True)
del df_combined, final_dfs
gc.collect()

print(f"\nRegenerated: {df_regen.shape}")
print(f"Match target 1,041,092: {len(df_regen) == 1_041_092}")
print(df_regen['Label'].value_counts())

Before capping: (13668745, 79)

Regenerated: (1041092, 79)
Match target 1,041,092: True
Label
Benign                      150000
DoS attacks-Hulk            150000
DDoS attacks-LOIC-HTTP      150000
DDOS attack-HOIC            150000
Infilteration               147427
Bot                         145460
SSH-Bruteforce               94075
DoS attacks-GoldenEye        41406
DoS attacks-Slowloris         9908
DDOS attack-LOIC-UDP          1730
Brute Force -Web               570
Brute Force -XSS               229
DoS attacks-SlowHTTPTest       107
FTP-BruteForce                  95
SQL Injection                   85
Name: count, dtype: int64


In [5]:
df_regen.to_parquet(CLEAN_DIR / "cic2018_final_sampled.parquet", index=False)
print("Saved. Verifying...")

df_check = pd.read_parquet(CLEAN_DIR / "cic2018_final_sampled.parquet")
print(f"On disk: {len(df_check):,} rows")
del df_check
gc.collect()

Saved. Verifying...
On disk: 1,041,092 rows


0

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

df = pd.read_parquet(CLEAN_DIR / "cic2018_final_sampled.parquet")

le = joblib.load(MODELS_DIR / "label_encoder.pkl")
df['label_encoded'] = le.transform(df['Label'])

X = df.drop(columns=['Label', 'label_encoded'])
y = df['label_encoded']

# Identical split to the XGBoost run — same random_state, same stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Test set: {X_test.shape}  (should be 208,219 rows)")

# ── Rule-based baseline ──
# Simulates a traditional signature/threshold SIEM rule set: static
# thresholds on well-known indicators, with no learning involved.
# Thresholds chosen from domain knowledge of each attack's behaviour.
BENIGN = list(le.classes_).index('Benign')
DDOS_HOIC = list(le.classes_).index('DDOS attack-HOIC')
DOS_HULK = list(le.classes_).index('DoS attacks-Hulk')
SSH_BF = list(le.classes_).index('SSH-Bruteforce')

def rule_based_classify(row):
    # Rule 1: very high packet rate → volumetric DDoS
    if row['Flow Pkts/s'] > 10000:
        return DDOS_HOIC
    # Rule 2: high byte rate with short flows → DoS flood
    if row['Flow Byts/s'] > 1_000_000 and row['Flow Duration'] < 1_000_000:
        return DOS_HULK
    # Rule 3: repeated small flows to port 22 → SSH brute force
    if row['Dst Port'] == 22 and row['Tot Fwd Pkts'] < 20:
        return SSH_BF
    # Default: treat as benign (mirrors real SIEMs — no rule match, no alert)
    return BENIGN

print("\nApplying rule-based baseline...")
y_pred_baseline = X_test.apply(rule_based_classify, axis=1)

print("\n=== RULE-BASED BASELINE ===")
print(f"Weighted F1:        {f1_score(y_test, y_pred_baseline, average='weighted', zero_division=0):.4f}")
print(f"Weighted Precision: {precision_score(y_test, y_pred_baseline, average='weighted', zero_division=0):.4f}")
print(f"Weighted Recall:    {recall_score(y_test, y_pred_baseline, average='weighted', zero_division=0):.4f}")

# ── XGBoost, same test set ──
model = joblib.load(MODELS_DIR / "xgb_model.pkl")
scaler = joblib.load(MODELS_DIR / "scaler.pkl")
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
y_pred_xgb = model.predict(X_test_scaled)

print("\n=== XGBOOST (AI MODEL) ===")
print(f"Weighted F1:        {f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0):.4f}")
print(f"Weighted Precision: {precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0):.4f}")
print(f"Weighted Recall:    {recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0):.4f}")

Test set: (208219, 78)  (should be 208,219 rows)

Applying rule-based baseline...

=== RULE-BASED BASELINE ===
Weighted F1:        0.0332
Weighted Precision: 0.0333
Weighted Recall:    0.1243

=== XGBOOST (AI MODEL) ===
Weighted F1:        0.9357
Weighted Precision: 0.9372
Weighted Recall:    0.9360


In [7]:
results_summary = pd.DataFrame({
    'Model': ['Rule-Based Baseline', 'XGBoost (AISecOpt)'],
    'Weighted F1': [0.0332, 0.9357],
    'Weighted Precision': [0.0333, 0.9372],
    'Weighted Recall': [0.1243, 0.9360]
})
OUTPUT_DIR = Path("../outputs/")
OUTPUT_DIR.mkdir(exist_ok=True)
results_summary.to_csv(OUTPUT_DIR / "baseline_vs_xgboost.csv", index=False)
print(results_summary.to_string(index=False))

              Model  Weighted F1  Weighted Precision  Weighted Recall
Rule-Based Baseline       0.0332              0.0333           0.1243
 XGBoost (AISecOpt)       0.9357              0.9372           0.9360
